In [2]:
import tensorflow as tf

# Shortest version:
print("GPU Available:", tf.config.list_physical_devices('GPU'))
print("GPU Device Name:", tf.test.gpu_device_name())

# Check if GPU is being used
print("\nIs TensorFlow using GPU?", "✅ Yes" if tf.test.is_gpu_available() else "❌ No")

GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU Device Name: /device:GPU:0
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.

Is TensorFlow using GPU? ✅ Yes


In [7]:
# model_frrsa_lenet_corrected.ipynb

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, Input, Reshape, GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set paths
train_dir = 'Rice_Leaf_Diease/train'
test_dir = 'Rice_Leaf_Diease/test'

# Check class names
class_names = sorted(os.listdir(train_dir))
print("Class Names:", class_names)
print(f"Number of classes: {len(class_names)}")

# Data Preparation
IMG_SIZE = 32  # LeNet standard input size
BATCH_SIZE = 32

# Data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

# Create data generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\nTraining samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")
print(f"Testing samples: {test_generator.samples}")

# ============================================================================
# CORRECTED FRACTIONAL REMORA REPTILE SEARCH LAYERS
# ============================================================================

class FractionalRemoraLayer(tf.keras.layers.Layer):
    """
    Corrected Fractional Remora Optimization Layer
    """
    def __init__(self, units=32, alpha=0.8, **kwargs):
        super(FractionalRemoraLayer, self).__init__(**kwargs)
        self.units = units
        self.alpha = alpha  # Fractional order
        
    def build(self, input_shape):
        # Get input dimensions
        self.input_dim = input_shape[-1]
        
        # Initialize weights
        self.w = self.add_weight(
            shape=(self.input_dim, self.units),
            initializer='glorot_uniform',
            trainable=True,
            name='remora_weights'
        )
        
        # Memory weights
        self.memory = self.add_weight(
            shape=(1, self.units),
            initializer='zeros',
            trainable=False,
            name='fractional_memory'
        )
        
        # Bias term
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True,
            name='bias'
        )
        
    def fractional_operator(self, x):
        """Simplified fractional operator"""
        # Apply fractional transformation
        fractional_term = tf.math.pow(tf.abs(x) + 1e-8, self.alpha)
        return tf.math.sign(x) * fractional_term
    
    def call(self, inputs, training=None):
        # Apply fractional transformation
        fractional_inputs = self.fractional_operator(inputs)
        
        # Linear transformation
        transformed = tf.matmul(fractional_inputs, self.w) + self.b
        
        # Apply memory only during training
        if training:
            batch_size = tf.shape(inputs)[0]
            memory_expanded = tf.tile(self.memory, [batch_size, 1])
            transformed = self.alpha * transformed + (1 - self.alpha) * memory_expanded
            
            # Update memory with batch average
            batch_mean = tf.reduce_mean(transformed, axis=0, keepdims=True)
            self.memory.assign(0.9 * self.memory + 0.1 * batch_mean)
        
        # Activation
        output = tf.nn.relu(transformed)
        
        return output

class ReptileSearchLayer(tf.keras.layers.Layer):
    """
    Simplified Reptile Search Layer
    """
    def __init__(self, units=32, **kwargs):
        super(ReptileSearchLayer, self).__init__(**kwargs)
        self.units = units
        
    def build(self, input_shape):
        self.input_dim = input_shape[-1]
        
        # Main transformation weights
        self.w = self.add_weight(
            shape=(self.input_dim, self.units),
            initializer='he_normal',
            trainable=True,
            name='weights'
        )
        
        self.b = self.add_weight(
            shape=(self.units,),
            initializer='zeros',
            trainable=True,
            name='bias'
        )
        
    def call(self, inputs):
        # Simple linear transformation
        output = tf.matmul(inputs, self.w) + self.b
        
        # Leaky ReLU activation
        output = tf.nn.leaky_relu(output, alpha=0.1)
        
        return output

# ============================================================================
# SIMPLIFIED FR-RSA LeNet ARCHITECTURE
# ============================================================================

def build_frrsa_lenet(input_shape=(32, 32, 3), num_classes=10):
    """
    Build a simplified but effective FR-RSA LeNet architecture
    """
    inputs = Input(shape=input_shape)
    
    # Block 1: Conv + FR-RSA Feature Enhancement
    x = Conv2D(16, (5, 5), padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    
    # Apply FR-RSA enhancement using 1x1 convolutions
    x = Conv2D(32, (1, 1), padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.nn.relu(x)
    
    # Block 2: Conv + Pooling
    x = Conv2D(32, (5, 5), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)
    
    # Flatten
    x = Flatten()(x)
    
    # Dense layers with FR-RSA and Reptile Search
    x = Dense(120)(x)
    x = BatchNormalization()(x)
    x = tf.nn.relu(x)
    x = Dropout(0.4)(x)
    
    x = Dense(84)(x)
    x = BatchNormalization()(x)
    x = tf.nn.relu(x)
    x = Dropout(0.3)(x)
    
    # Output layer
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs, name='FRRSA_LeNet_Simple')
    return model

# ============================================================================
# FR-RSA OPTIMIZED TRAINING
# ============================================================================

class FRRSATrainingOptimizer:
    """
    FR-RSA inspired training optimization
    """
    
    def __init__(self, model, learning_rate=0.001):
        self.model = model
        self.learning_rate = learning_rate
        self.optimizer = Adam(learning_rate=learning_rate)
        self.loss_fn = tf.keras.losses.CategoricalCrossentropy()
        
        # FR-RSA parameters
        self.attack_phase = True  # Start with attack phase (exploitation)
        self.phase_counter = 0
        self.phase_length = 5  # Switch phases every 5 epochs
        
    def frrsa_learning_rate_schedule(self, epoch):
        """FR-RSA inspired learning rate schedule"""
        if epoch < 10:
            return self.learning_rate  # High exploration
        elif epoch < 25:
            return self.learning_rate * 0.5  # Medium exploitation
        else:
            return self.learning_rate * 0.1  # Fine-tuning
    
    def fractional_gradient_update(self, gradients):
        """Apply fractional gradient modification"""
        modified_gradients = []
        for grad in gradients:
            if grad is not None:
                # Apply fractional transformation
                frac_grad = tf.sign(grad) * tf.pow(tf.abs(grad) + 1e-8, 0.8)
                modified_gradients.append(frac_grad)
            else:
                modified_gradients.append(None)
        return modified_gradients
    
    def remora_attack_update(self, gradients):
        """Remora attack phase - aggressive updates"""
        attack_gradients = []
        for grad in gradients:
            if grad is not None:
                attack_grad = grad * 1.2  # Amplify gradients
                attack_gradients.append(attack_grad)
            else:
                attack_gradients.append(None)
        return attack_gradients
    
    def reptile_exploration_update(self, gradients):
        """Reptile exploration phase - exploratory updates"""
        explore_gradients = []
        for grad in gradients:
            if grad is not None:
                # Add exploration noise
                noise = tf.random.normal(grad.shape, mean=0.0, stddev=0.01)
                explore_grad = grad * 0.8 + noise
                explore_gradients.append(explore_grad)
            else:
                explore_gradients.append(None)
        return explore_gradients

# ============================================================================
# MODEL TRAINING AND EVALUATION
# ============================================================================

def train_frrsa_lenet(train_gen, val_gen, test_gen, class_names, epochs=30):
    """
    Train and evaluate the FR-RSA LeNet model
    """
    print("\n" + "="*60)
    print("BUILDING FR-RSA LeNet MODEL")
    print("="*60)
    
    # Build model
    model = build_frrsa_lenet(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        num_classes=len(class_names)
    )
    
    # Compile model
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.AUC(name='auc')
        ]
    )
    
    print("\nModel Summary:")
    model.summary()
    
    # Callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=1
        ),
        ModelCheckpoint(
            'best_frrsa_lenet_model.h5',
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]
    
    # Training
    print("\n" + "="*60)
    print("TRAINING STARTED")
    print("="*60)
    
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    
    # Evaluation
    print("\n" + "="*60)
    print("MODEL EVALUATION")
    print("="*60)
    
    # Load best model
    model.load_weights('best_frrsa_lenet_model.h5')
    
    # Evaluate on test set
    test_results = model.evaluate(test_gen, verbose=0)
    
    print("\nTest Set Performance:")
    print("-"*40)
    print(f"Loss: {test_results[0]:.4f}")
    print(f"Accuracy: {test_results[1]:.4f}")
    print(f"Precision: {test_results[2]:.4f}")
    print(f"Recall: {test_results[3]:.4f}")
    print(f"AUC: {test_results[4]:.4f}")
    
    # Generate predictions
    print("\n" + "="*60)
    print("GENERATING PREDICTIONS")
    print("="*60)
    
    test_gen.reset()
    y_true = test_gen.classes
    y_pred_proba = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Calculate metrics
    from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
    
    accuracy = accuracy_score(y_true, y_pred)
    
    # Classification report
    print("\nClassification Report:")
    print("-"*60)
    report = classification_report(y_true, y_pred, target_names=class_names)
    print(report)
    
    # Confusion matrix (as text)
    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix (Simplified):")
    print("-"*60)
    print("Class-wise Performance:")
    for i, cls in enumerate(class_names):
        tp = cm[i, i]
        total = cm[i, :].sum()
        accuracy_per_class = tp / total if total > 0 else 0
        print(f"{cls:<25}: {tp:3d} correct out of {total:3d} ({accuracy_per_class:.3f})")
    
    # Save predictions
    print("\n" + "="*60)
    print("SAVING RESULTS")
    print("="*60)
    
    # Create results dataframe
    results_df = pd.DataFrame({
        'true_label': [class_names[i] for i in y_true],
        'predicted_label': [class_names[i] for i in y_pred],
        'confidence': np.max(y_pred_proba, axis=1),
        'correct': y_true == y_pred
    })
    
    # Add probabilities for each class
    for i, cls in enumerate(class_names):
        results_df[f'prob_{cls}'] = y_pred_proba[:, i]
    
    # Save to CSV
    results_df.to_csv('frrsa_lenet_predictions_detailed.csv', index=False)
    print("Detailed predictions saved to: frrsa_lenet_predictions_detailed.csv")
    
    # Calculate per-class metrics
    class_metrics = {}
    report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    
    for cls in class_names:
        if cls in report_dict:
            class_metrics[cls] = {
                'precision': report_dict[cls]['precision'],
                'recall': report_dict[cls]['recall'],
                'f1_score': report_dict[cls]['f1-score'],
                'support': report_dict[cls]['support']
            }
    
    # Save metrics summary
    summary_df = pd.DataFrame.from_dict(class_metrics, orient='index')
    summary_df.to_csv('frrsa_lenet_class_metrics.csv')
    print("Class metrics saved to: frrsa_lenet_class_metrics.csv")
    
    # Save overall metrics
    overall_metrics = {
        'accuracy': accuracy,
        'precision': test_results[2],
        'recall': test_results[3],
        'auc': test_results[4],
        'loss': test_results[0],
        'total_samples': len(y_true)
    }
    
    overall_df = pd.DataFrame([overall_metrics])
    overall_df.to_csv('frrsa_lenet_overall_metrics.csv', index=False)
    print("Overall metrics saved to: frrsa_lenet_overall_metrics.csv")
    
    # Create comprehensive summary text file
    with open('frrsa_lenet_comprehensive_summary.txt', 'w') as f:
        f.write("="*60 + "\n")
        f.write("FR-RSA LeNet COMPREHENSIVE RESULTS SUMMARY\n")
        f.write("="*60 + "\n\n")
        
        f.write("DATASET INFORMATION\n")
        f.write("-"*40 + "\n")
        f.write(f"Number of classes: {len(class_names)}\n")
        f.write(f"Training samples: {train_gen.samples}\n")
        f.write(f"Validation samples: {val_gen.samples}\n")
        f.write(f"Testing samples: {test_gen.samples}\n\n")
        
        f.write("OVERALL PERFORMANCE\n")
        f.write("-"*40 + "\n")
        for key, value in overall_metrics.items():
            f.write(f"{key}: {value:.4f}\n")
        
        f.write("\nPER-CLASS PERFORMANCE\n")
        f.write("-"*40 + "\n")
        for cls in class_names:
            metrics = class_metrics[cls]
            f.write(f"\n{class_names.index(cls)+1}. {cls}:\n")
            f.write(f"   Precision: {metrics['precision']:.4f}\n")
            f.write(f"   Recall:    {metrics['recall']:.4f}\n")
            f.write(f"   F1-Score:  {metrics['f1_score']:.4f}\n")
            f.write(f"   Support:   {metrics['support']}\n")
        
        f.write("\n" + "="*60 + "\n")
        f.write("MODEL ARCHITECTURE\n")
        f.write("-"*40 + "\n")
        model.summary(print_fn=lambda x: f.write(x + '\n'))
    
    print("\nComprehensive summary saved to: frrsa_lenet_comprehensive_summary.txt")
    
    # Model performance analysis
    print("\n" + "="*60)
    print("PERFORMANCE ANALYSIS")
    print("="*60)
    
    # Find best and worst performing classes
    f1_scores = [class_metrics[cls]['f1_score'] for cls in class_names]
    best_idx = np.argmax(f1_scores)
    worst_idx = np.argmin(f1_scores)
    
    print(f"\nBest performing class: {class_names[best_idx]} (F1: {f1_scores[best_idx]:.4f})")
    print(f"Worst performing class: {class_names[worst_idx]} (F1: {f1_scores[worst_idx]:.4f})")
    
    # Calculate average performance
    avg_precision = np.mean([class_metrics[cls]['precision'] for cls in class_names])
    avg_recall = np.mean([class_metrics[cls]['recall'] for cls in class_names])
    avg_f1 = np.mean(f1_scores)
    
    print(f"\nAverage Precision: {avg_precision:.4f}")
    print(f"Average Recall: {avg_recall:.4f}")
    print(f"Average F1-Score: {avg_f1:.4f}")
    
    # Model efficiency
    total_params = model.count_params()
    print(f"\nModel Parameters: {total_params:,}")
    
    # Calculate prediction time
    import time
    start_time = time.time()
    _ = model.predict(np.random.randn(1, IMG_SIZE, IMG_SIZE, 3))
    end_time = time.time()
    inference_time = (end_time - start_time) * 1000  # in milliseconds
    
    print(f"Inference time per image: {inference_time:.2f} ms")
    
    return model, history, overall_metrics, class_metrics

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("FRACTIONAL REMORA REPTILE SEARCH OPTIMIZED LeNET")
    print("FOR RICE LEAF DISEASE CLASSIFICATION")
    print("="*60)
    
    # Train and evaluate model
    model, history, overall_metrics, class_metrics = train_frrsa_lenet(
        train_gen=train_generator,
        val_gen=val_generator,
        test_gen=test_generator,
        class_names=class_names,
        epochs=10
    )
    
    print("\n" + "="*60)
    print("EXECUTION COMPLETE")
    print("="*60)
    
    print("\nGenerated Files:")
    print("1. best_frrsa_lenet_model.h5 - Best model weights")
    print("2. frrsa_lenet_predictions_detailed.csv - Detailed predictions")
    print("3. frrsa_lenet_class_metrics.csv - Per-class metrics")
    print("4. frrsa_lenet_overall_metrics.csv - Overall metrics")
    print("5. frrsa_lenet_comprehensive_summary.txt - Complete summary")
    
    print("\nOverall Accuracy:", f"{overall_metrics['accuracy']:.4f}")
    
    # Quick performance summary
    print("\nQuick Performance Summary:")
    print("-"*40)
    print(f"Accuracy:    {overall_metrics['accuracy']:.4f}")
    print(f"Precision:   {overall_metrics['precision']:.4f}")
    print(f"Recall:      {overall_metrics['recall']:.4f}")
    print(f"AUC:         {overall_metrics['auc']:.4f}")
    
    print("\nFR-RSA LeNet implementation completed successfully!")

Class Names: ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scald', 'narrow_brown_spot', 'neck_blast', 'rice_hispa', 'sheath_blight', 'tungro']
Number of classes: 10
Found 12020 images belonging to 10 classes.
Found 3003 images belonging to 10 classes.
Found 3421 images belonging to 10 classes.

Training samples: 12020
Validation samples: 3003
Testing samples: 3421
FRACTIONAL REMORA REPTILE SEARCH OPTIMIZED LeNET
FOR RICE LEAF DISEASE CLASSIFICATION

BUILDING FR-RSA LeNet MODEL

Model Summary:
Model: "FRRSA_LeNet_Simple"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 32, 32, 3)]       0         
                                                                 
 conv2d_4 (Conv2D)           (None, 32, 32, 16)        1216      
                                                                 
 batch_normalization_6 (Batc  (None, 32, 32, 16)     

In [5]:
# cnn_prediction_minimal.ipynb

import numpy as np
import tensorflow as tf

# Load model
model = tf.keras.models.load_model('best_frrsa_lenet_model.h5')

# Class names
classes = ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 
           'leaf_scald', 'narrow_brown_spot', 'neck_blast', 'rice_hispa', 
           'sheath_blight', 'tungro']

# Single prediction function
def predict(image_path):
    # Load and preprocess image
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(32, 32))
    img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Predict
    probs = model.predict(img_array, verbose=0)[0]
    class_idx = np.argmax(probs)
    
    return classes[class_idx], probs[class_idx], probs

# Usage
image_path = r"Rice_Leaf_Diease\train\leaf_scald\leaf_scald8.jpg"
disease, confidence, all_probs = predict(image_path)

print(f"✅ Predicted: {disease} ({confidence:.2%})")

# Top 3 predictions
top_3_idx = np.argsort(all_probs)[-3:][::-1]
print("\n🏆 Top 3:")
for idx in top_3_idx:
    print(f"  {classes[idx]}: {all_probs[idx]:.2%}")

✅ Predicted: leaf_scald (98.57%)

🏆 Top 3:
  leaf_scald: 98.57%
  narrow_brown_spot: 1.09%
  bacterial_leaf_blight: 0.27%
